<a href="https://colab.research.google.com/github/Keerthan-AC-583/BIS-Lab/blob/main/1BM24CS133_GWO_EV_charging_placement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# 1. Define the Problem Parameters & Objective Function
np.random.seed(42)
NUM_DEMAND_CENTERS = 15
# Generate random city demand centers (X, Y) and their weights (demand load)
DEMAND_LOCS = np.random.uniform(0, 100, (NUM_DEMAND_CENTERS, 2))
DEMAND_WEIGHTS = np.random.randint(10, 100, NUM_DEMAND_CENTERS)

def objective_function(wolf_position):
    """
    Evaluates an EVCS configuration.
    Aims to minimize: (Sum of weighted distances to the nearest station) + (Fixed Installation Cost)
    """
    # Reshape 1D wolf array back to (M, 2) station locations
    stations = wolf_position.reshape(-1, 2)
    total_travel_cost = 0

    # Calculate travel distance cost to the closest station for all demand nodes
    for i in range(NUM_DEMAND_CENTERS):
        distances = np.linalg.norm(stations - DEMAND_LOCS[i], axis=1)
        min_dist = np.min(distances)
        total_travel_cost += min_dist * DEMAND_WEIGHTS[i]

    # Example fixed infrastructure cost component (e.g., penalties for overlapping locations)
    station_distances = np.linalg.norm(stations[:, None, :] - stations[None, :, :], axis=-1)
    overlap_penalty = np.sum(station_distances < 5) - len(stations) # penalty if stations are too close

    return total_travel_cost + (overlap_penalty * 500)

# 2. Grey Wolf Optimizer implementation
def grey_wolf_optimizer(num_wolves, max_iter, num_stations, lb, ub):
    # Dimensions: each station has 2 variables (X, Y coords)
    dim = num_stations * 2

    # Initialize wolves randomly within bounds
    wolves = np.random.uniform(lb, ub, (num_wolves, dim))

    # Initialize alpha, beta, and delta scores/positions
    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    alpha_pos = np.zeros(dim)
    beta_pos = np.zeros(dim)
    delta_pos = np.zeros(dim)

    # Main optimization loop
    for t in range(max_iter):
        for i in range(num_wolves):
            # Clip variables to remain within geographical boundaries
            wolves[i] = np.clip(wolves[i], lb, ub)

            # Calculate fitness
            fitness = objective_function(wolves[i])

            # Update Alpha, Beta, and Delta hierarchies
            if fitness < alpha_score:
                alpha_score = fitness
                alpha_pos = wolves[i].copy()
            elif fitness < beta_score:
                beta_score = fitness
                beta_pos = wolves[i].copy()
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = wolves[i].copy()

        # Linearly decrease adaptive parameter 'a' from 2 to 0
        a = 2 - t * (2 / max_iter)

        # Update positions of all wolves
        for i in range(num_wolves):
            new_position = np.zeros(dim)
            for j in range(dim):
                # Calculations relative to Alpha wolf
                r1, r2 = np.random.random(), np.random.random()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - wolves[i][j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Calculations relative to Beta wolf
                r1, r2 = np.random.random(), np.random.random()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - wolves[i][j])
                X2 = beta_pos[j] - A2 * D_beta

                # Calculations relative to Delta wolf
                r1, r2 = np.random.random(), np.random.random()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - wolves[i][j])
                X3 = delta_pos[j] - A3 * D_delta

                # Hunting encircle and convergence formula
                new_position[j] = (X1 + X2 + X3) / 3

            wolves[i] = new_position

        if (t + 1) % 10 == 0 or t == 0:
            print(f"Iteration {t+1}/{max_iter} | Best Cost (Alpha Score): {alpha_score:.2f}")

    return alpha_pos.reshape(-1, 2), alpha_score

# 3. Execution Execution
NUM_WOLVES = 30
MAX_ITERATIONS = 50
NUM_STATIONS = 3
LOWER_BOUND = 0
UPPER_BOUND = 100

best_coords, best_cost = grey_wolf_optimizer(NUM_WOLVES, MAX_ITERATIONS, NUM_STATIONS, LOWER_BOUND, UPPER_BOUND)

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Optimal Total Cost: {best_cost:.2f}")
print("Optimal Coordinates for Charging Stations:")
for index, coordinate in enumerate(best_coords, start=1):
    print(f" Station {index}: X = {coordinate[0]:.2f}, Y = {coordinate[1]:.2f}")


Iteration 1/50 | Best Cost (Alpha Score): 16376.86
Iteration 10/50 | Best Cost (Alpha Score): 14043.07
Iteration 20/50 | Best Cost (Alpha Score): 13029.38
Iteration 30/50 | Best Cost (Alpha Score): 12881.68
Iteration 40/50 | Best Cost (Alpha Score): 12551.78
Iteration 50/50 | Best Cost (Alpha Score): 12262.48

=== OPTIMIZATION COMPLETE ===
Optimal Total Cost: 12262.48
Optimal Coordinates for Charging Stations:
 Station 1: X = 29.18, Y = 36.65
 Station 2: X = 59.72, Y = 1.50
 Station 3: X = 35.97, Y = 85.59
